# Notebook 4: Agentic RAG Introduction

In this notebook you will learn how to classify a user search and have specialize agents called depending on the user's question. This notebook does not use the Azure AI Search index to create a full solution for you - you will get to do that in the hands on exercise.

## Learning Objectives
- Learn about the WorkflowBuilder and conditional routing using switch-case logic
- Implement a classifier to recognize the user's intent and route to a specialized agent
- Implement agents to respond to specific search types


### Setup the Module Imports

In [1]:
import os

from collections.abc import AsyncIterable
from typing import Annotated, cast, Literal
from pydantic import BaseModel

from agent_framework import (
    Agent,
    AgentResponse,
    AgentExecutorResponse,
    Default,
    Case,
    Executor,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowRunResult,
    executor,
    handler,
    tool,
)
from agent_framework.openai import OpenAIChatClient
from agent_framework.orchestrations import HandoffAgentUserRequest, HandoffBuilder
from azure.identity import DefaultAzureCredential

Define AI functions you'll use for testing the specific search type in this notebook.

> NOTE: these are where the actual implementation goes for doing specific search types in the hands on notebook

In [2]:
@tool(approval_mode="never_require")
def yes_or_no_search(user_question: Annotated[str, "User question to answer with yes or no"]) -> str:
    """Answers yes or no questions."""
    return f"Question: {user_question}\nAnswer: NO"

@tool(approval_mode="never_require")
def count_search(user_question: Annotated[str, "User question requiring counting items"]) -> str:
    """Answers questinos that required counting."""
    return f"Question: {user_question}\nAnswer: 42"

@tool(approval_mode="never_require")
def semantic_search(user_question: Annotated[str, "User question requiring semantic search"]) -> str:
    """Answers simple question that requires semantic search."""
    return f"Question: {user_question}\nAnswer: They all the gray or metallic in color."

Create a prompt for the classifier to be able to determine how to route the user's question.

In [3]:
CLASSIFIER_AGENT_INSTRUCTIONS = """
You are a query classification system for an IT support ticket database.
   Classify the incoming user question into exactly one category and return
   a JSON object with "category" and "reasoning" fields.

   ## Database Schema
   The database contains IT support tickets with these fields:
   - Id: unique identifier
   - Create_Date: date the ticket was created
   - Subject: ticket subject
   - Body: ticket question/description
   - Answer: ticket response/solution
   - Type: ticket type (values: "Incident", "Request", "Problem", "Change")
   - Queue: department name (values: "Human Resources", "IT", "Finance", "Operations", "Sales", "Marketing", "Engineering", "Support")
   - Priority: priority level (values: "high", "medium", "low")
   - Language: ticket language
   - Business_Type: business category
   - Tags: categorization tags

   **IMPORTANT**: When "and" combines field values (Type, Queue, Priority), these are FILTERS for counting/searching, NOT separate items to compare.

   ## Categories (use these exact values):

   **difference**: Questions with negation/exclusion ("not", "don't", "without", "excluding").
      - "Which Dell XPS Issue does not mention Windows?" -> difference
      - "Find tickets without high priority" -> difference

   **intersection**: Questions combining multiple SEARCH TOPICS with "and", "both", "that also".
      - "What issues are for Dell XPS laptops and the user tried Win + Ctrl + Shift + B?" -> intersection
      - NOT when "and" combines field filters like Priority/Queue/Type.

   **multi_hop**: Questions asking for a different attribute than what's searched (find X, extract Y).
      - "What department had consultants with Login Issues?" -> multi_hop
      - "Which queue handles password reset requests?" -> multi_hop

   **comparative**: Questions comparing multiple items ("more", "less", "vs", "or").
      - "Do we have more issues with MacBook Air or Dell XPS?" -> comparative

   **yes_no**: Explicit yes/no questions expecting a boolean answer.
      - "Are there any issues for Dell XPS laptops?" -> yes_no

   **count**: Counting questions ("how many", "count", "total", "number of").
      - "How many Incidents for Human Resources and low priority?" -> count (all filters!)

   **semantic_search**: General queries about issues, solutions, how-to (everything else).
      - "What problems are there with Surface devices?" -> semantic_search

   ## Classification Priority
   1. COUNTING first: "how many", "count", "total" -> count
   2. NEGATION: "not", "don't", "without" -> difference
   3. COMPARISON: "more", "less", "vs", "or" comparing items -> comparative
   4. INTERSECTION: multiple search topics with "and" (NOT field filters) -> intersection
   5. MULTI-HOP: "What [FIELD] had [CONDITION]" -> multi_hop
   6. YES/NO: explicit boolean questions -> yes_no
   7. Everything else -> semantic_search

   ## Key Rules
   - Field values (Priority, Queue, Type) are FILTERS, not search topics
   - "How many X and Y and Z?" = count (filters). "What X and Y?" = intersection (topics)
   - "Which X does not mention Y?" = difference, NOT count
"""

### Define the Classification Models and Executors

The code below sets up the data flow between the classifier agent and the specialist agents:

1. **`ClassifyResult`** — A Pydantic model that represents the structured JSON output from the classifier agent (category + reasoning).
2. **`ClassifiedQuery`** — Carries the classification result together with the original user question so it can be routed downstream.
3. **`extract_category` executor** — Parses the classifier agent's JSON response, logs the classification, and forwards a `ClassifiedQuery` message to the next step in the workflow.
4. **`QueryBridge` executor** — A simple adapter that converts a `ClassifiedQuery` back into a plain string so the specialist agents receive just the user's question.

In [4]:
CATEGORY_LITERAL = Literal[
    "yes_no",
    "semantic_search",
    "count",
    "difference",
    "intersection",
    "multi_hop",
    "comparative",
]


class ClassifyResult(BaseModel):
    """Structured classification result returned by the classifier agent."""

    category: CATEGORY_LITERAL
    reasoning: str


class ClassifiedQuery(BaseModel):
    """Carries both the classification and original user question for routing."""

    category: CATEGORY_LITERAL
    user_question: str


# ---------------------------------------------------------------------------
# Executor: extract the structured classification and forward it
# ---------------------------------------------------------------------------

@executor(id="extract_category")
async def extract_category(
    response: AgentExecutorResponse,
    ctx: WorkflowContext[ClassifiedQuery],
) -> None:
    """Parse the classifier's structured JSON output and send it downstream."""
    text = response.agent_response.text.strip()

    # Strip markdown code fences if present
    if text.startswith("```"):
        text = text.split("\n", 1)[-1].rsplit("```", 1)[0].strip()

    # Extract the first complete JSON object in case of trailing characters
    brace_depth = 0
    json_end = -1
    for i, ch in enumerate(text):
        if ch == "{":
            brace_depth += 1
        elif ch == "}":
            brace_depth -= 1
            if brace_depth == 0:
                json_end = i + 1
                break
    if json_end > 0:
        text = text[:json_end]

    result = ClassifyResult.model_validate_json(text)
    print(f"  [Classified as: {result.category} -- {result.reasoning}]")

    # Retrieve the original user question from the conversation history
    user_question = next(
        (m.text for m in response.full_conversation if m.role == "user" and m.text),
        "",
    )
    await ctx.send_message(
        ClassifiedQuery(category=result.category, user_question=user_question)
    )


# ---------------------------------------------------------------------------
# Bridge executor: convert ClassifiedQuery -> str for specialist agents
# ---------------------------------------------------------------------------

class QueryBridge(Executor):
    """Converts a ClassifiedQuery into a plain string for a downstream agent."""

    @handler
    async def forward(
        self, query: ClassifiedQuery, ctx: WorkflowContext[str]
    ) -> None:
        await ctx.send_message(query.user_question)


### Create the Agents



In [5]:
def create_agents(chat_client: OpenAIChatClient) -> tuple[Agent, Agent, Agent, Agent]:
    """Create and configure the classifier and search specialist agents.

    The classifier agent is responsible for:
    - Receiving all user input first
    - Deciding whether to handle the request directly or hand off to a search specialist
    - Signaling handoff by calling one of the explicit handoff tools exposed to it

    Search specialist agents are invoked only when the classifier agent explicitly hands off to them.
    After a specialist responds, control returns to the classifier agent, which then prompts
    the user for their next message.

    Returns:
        Tuple of (classifier_agent, yes_no_agent, count_agent, semantic_search_agent)
    """
    # Classifier agent: Acts as the search planner and dispatcher
    classifier_agent = chat_client.as_agent(
        instructions=CLASSIFIER_AGENT_INSTRUCTIONS,
        name="classifier_agent",
        default_options={"response_format": ClassifyResult},
        require_per_service_call_history_persistence=True,
    )

    # Yes/No search specialist: Handles yes/no questions
    yes_no_agent = chat_client.as_agent(
        instructions="You handle yes/no questions. When done, hand off back to the classifier_agent.",
        name="yes_no_agent",
        tools=[yes_or_no_search],
        require_per_service_call_history_persistence=True,
    )

    # Count search specialist: Handles counting questions
    count_agent = chat_client.as_agent(
        instructions="You handle questions that require counting items. When done, hand off back to the classifier_agent.",
        name="count_agent",
        tools=[count_search],
        require_per_service_call_history_persistence=True,
    )

    # Semantic search specialist: Handles semantic search questions
    semantic_search_agent = chat_client.as_agent(
        instructions="You handle questions that require semantic search. When done, hand off back to the classifier_agent.",
        name="semantic_search_agent",
        tools=[semantic_search],
        require_per_service_call_history_persistence=True,
    )

    return classifier_agent, yes_no_agent, count_agent, semantic_search_agent

### Build the Workflow with Handoff Routing


In [6]:
def build_workflow(agents):
    """
    Build a WorkflowBuilder workflow with switch-case routing.

    Pipeline:
        classifier_agent -> extract_category -> [switch_case] -> bridge -> specialist_agent
    """
    classifier = agents["classifier"]

    # Create bridge executors (one per specialist) to convert
    # ClassifiedQuery -> str for the downstream agent executors.
    bridges = {
        name: QueryBridge(id=f"{name}_bridge")
        for name in [
            "yes_no", "semantic_search", "count",
        ]
    }

    builder = (
        WorkflowBuilder(name="agentic_rag_workflow", start_executor=classifier)
        .add_edge(classifier, extract_category)
        .add_switch_case_edge_group(
            extract_category,
            [
                Case(condition=lambda r: isinstance(r, ClassifiedQuery) and r.category == "yes_no",
                     target=bridges["yes_no"]),
                Case(condition=lambda r: isinstance(r, ClassifiedQuery) and r.category == "count",
                     target=bridges["count"]),
                Default(target=bridges["semantic_search"]),
            ],
        )
    )

    # Wire each bridge to its specialist agent
    for name, bridge in bridges.items():
        builder = builder.add_edge(bridge, agents[name])

    return builder.build()

### Initialize the Client, Agents, and Workflow

Load environment variables, create the Azure OpenAI chat client, instantiate all agents (classifier + specialists), and build the workflow. After this cell runs, the `workflow` object is ready to process user questions.

In [7]:
import dotenv
dotenv.load_dotenv()

# Initialize the Azure OpenAI chat client
chat_client = OpenAIChatClient(
    model=os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"),
    credential=DefaultAzureCredential()
)

# Create all agents: classifier + specialists
classifier, yes_no, count, semantic_search = create_agents(chat_client)

agents = {
    "classifier": classifier,
    "yes_no": yes_no,
    "count": count,
    "semantic_search": semantic_search,
}

workflow = build_workflow(agents)

For demo purposes create a list of some sample questions.

In [8]:
demo_questions  = [
    "How many tickets were logged and Incidents for Human Resources and low priority?", # count search
    "Are there any issues logged for Dell XPS laptops?" # yes/no search
]

### Run the Workflow

In [9]:
def handle_workflow_result(result: WorkflowRunResult) -> None:
    """
    Process a WorkflowRunResult and print the outputs.

    Iterates over the workflow outputs and prints agent responses.

    Args:
        result: WorkflowRunResult returned by workflow.run()
    """
    outputs = result.get_outputs()

    if not outputs:
        print("[No output produced by the workflow]")
        return

    for output in outputs:
        if isinstance(output, AgentResponse):
            for message in output.messages:
                if not message.text:
                    continue
                speaker = message.author_name or message.role
                print(f"  {speaker}: {message.text}")
        elif isinstance(output, str):
            print(f"  {output}")
        else:
            print(f"  {output}")

Loop through the demo questions, run each one through the workflow, and print the results. Each run is independent — the `WorkflowBuilder` workflow is stateless per run, so no conversation history carries over between questions.

In [10]:
for i, question in enumerate(demo_questions, 1):
    print(f"\n--- Query {i}/{len(demo_questions)} ---")
    print(f"User: {question}\n")

    # Each run is independent -- WorkflowBuilder workflows are stateless per run
    result = await workflow.run(question)
    handle_workflow_result(result)


--- Query 1/2 ---
User: How many tickets were logged and Incidents for Human Resources and low priority?

  [Classified as: count -- The question asks 'How many tickets' indicating a counting query, and applies filters for ticket Type (Incidents), Queue (Human Resources), and Priority (low). This is a standard count query applying multiple filters.]
  classifier_agent: {"category":"count","reasoning":"The question asks 'How many tickets' indicating a counting query, and applies filters for ticket Type (Incidents), Queue (Human Resources), and Priority (low). This is a standard count query applying multiple filters."}
  count_agent: There were 42 tickets logged for Human Resources with low priority, and also 42 Incidents logged for Human Resources with low priority.

--- Query 2/2 ---
User: Are there any issues logged for Dell XPS laptops?

  [Classified as: yes_no -- The question starts with 'Are there any' indicating a yes/no question asking if issues exist for Dell XPS laptops.]
  c

As I've pointed out in the other notebooks, the question "How many tickets were logged and Incidents for Human Resources and low priority?" is a hard one to get the system to answer - and this time the classifier correctly identified it as something the count_agent should take care of. That is a start!

Next is the hands on exercise where you get to turn these lessons from the notebooks into an Agentic RAG solution for the IT support search index we've been using.